<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# My Response:
One row represents the daily performance of one content item for one pseudonymized client on one report date.

For development, I will use the March 2026 partition rather than the final June 2026 month. This keeps the final month available as a sealed test period. My decision use case is content opportunity scoring: prioritizing content items for review

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# My Response:
**Feature:**

I will initially use five performance features:

- `gsc_impressions:` available from Search Console performance data for the observation period before the decision is made.
- `gsc_clicks:` available from Search Console performance data for the observation period before the decision is made.
- `gsc_avg_position:` available from Search Console performance data for the observation period before the decision is made.
- `ga4_sessions:` available for rows where ga4_data_available IS TRUE; otherwise GA4 values are not treated as genuine zero engagement.
- `ga4_engaged_sessions:` available for rows where `ga4_data_available` IS TRUE; otherwise GA4 values are not treated as genuine zero engagement

**Label/proxy:**

 A future decline or opportunity proxy will be defined only after the relevant historical and future windows are verified. I will not use current-period outcome fields as features if they are used to construct the target.

 **Context fields:**

- report_date
- client_hash_id
- content_hash_id
- gsc_data_available
- ga4_data_available

These fields provide information about the observation, identity, date, or data availability but are not treated as predictive features.

**Deliberately excluded:**

`client_hash_id` and `content_hash_id` will not be used as model features because they are pseudonymous identifiers rather than meaningful behavioral signals. I will also exclude any field that is derived from, or contains information from, the future outcome window to prevent leakage. GA4 metrics will only be used when ga4_data_available IS TRUE; zero-filled values before GA4 availability will not be interpreted as genuine zero engagement.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Warehouse connection

In [46]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

print("Connected")

Connected


The warehouse is hosted on Hugging Face. I use DuckDB to query the Parquet partitions directly rather than loading the full warehouse into memory.

The Hugging Face access token is stored in Colab Secrets as `HF_TOKEN` and is not written into the notebook.

The connection succeeded, so the notebook can query the warehouse directly from the March 2026 partition.

In [47]:
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"
)

I use the March 2026 partition as the development window. The warehouse documentation recommends using a mid-panel month rather than the final `_sample` month, because the final month can represent the natural outcome window for a future-looking label.

Using a fixed monthly partition also keeps the queries smaller and makes the analysis reproducible.

### **Query 1:** Verify the grain

The expected grain is one row per:

`report_date × client_hash_id × content_hash_id`

I group by these three fields and search for combinations appearing more than once.


In [48]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


No duplicate `report_date × client_hash_id × content_hash_id` combinations were found in the March 2026 partition. This supports the stated daily performance grain: one row represents one content item for one client on one report date.

###**Query 2:** Row count and date span

In [49]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{march_path}')
""").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


The March 2026 partition contains **9,841,378 rows**, spanning **2026-03-01 through 2026-03-31**.

The query confirms that the selected partition covers the complete March calendar month represented in the warehouse. The large row count is consistent with a daily content-performance table because the same content/client combinations can appear across multiple report dates.

###**Query 3:**  Data Availability

In [50]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


The availability flags show how much of the March partition has usable data from each source.

- **GSC available:** 3,611,061 of 9,841,378 rows (**36.7%**)
- **GA4 available:** 413,966 of 9,841,378 rows (**4.2%**)

  
GSC coverage is substantially higher than GA4 coverage in this March slice. Only a small proportion of rows have GA4 data available, so GA4-based features cannot be assumed to exist for every observation.

This also means that a missing or zero-filled GA4 value must not automatically be interpreted as zero engagement. The `ga4_data_available` flag needs to be considered when constructing GA4-based features.

# Leakage Experiment

To demonstrate leakage on real warehouse data, I use April 2026 as a future outcome window following the March 2026 decision period.

The purpose of this experiment is not to build the final FlyRank model. Instead, it deliberately introduces one feature that would not be available at the March decision moment and observes how much the evaluation score changes.

Before constructing the proxy label, I verify that the April partition covers 2026-04-01 through 2026-04-30 and inspect GSC availability.

In [51]:
april_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet"
)

con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{april_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,10424730,2026-04-01,2026-04-30


In [52]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet('{april_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,10424730,3901060


The April partition contains 10,424,730 rows and spans 2026-04-01 through 2026-04-30.

Of these rows, 3,901,060 have GSC data available. This confirms that April provides a future GSC observation window for the leakage demonstration.

### March decision-period snapshot

In [53]:
march_snapshot = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,
    AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position

FROM read_parquet('{march_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [54]:
march_snapshot.head()

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.499519


The daily March performance data is aggregated to one row per `client_hash_id × content_hash_id`.

The resulting snapshot contains three GSC-based performance features:

- total March impressions
- total March clicks
- average March position

These represent information available during the March observation period and are therefore treated as the legitimate feature set for this experiment.

I use GSC-only features here because GA4 availability is substantially lower in the March slice, and I want the leakage demonstration to avoid treating unavailable GA4 values as genuine zero engagement.

### April outcome snapshot


In [55]:
april_snapshot = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS april_impressions

FROM read_parquet('{april_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

april_snapshot.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,april_impressions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


April is treated as the future observation window.

I aggregate April impressions to the same `client_hash_id × content_hash_id` grain so that each March decision-period observation can be matched to its subsequent April outcome.

April information is used only to construct the experimental outcome and is not available to the model in the honest baseline.
Only rows where `gsc_data_available IS TRUE` are included when calculating the April impression outcome.

### March – April matched dataset

In [56]:
df = march_snapshot.merge(
    april_snapshot,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

df.shape

(158549, 6)

The March and April snapshots are joined on `client_hash_id` and `content_hash_id`.

The resulting dataset contains **158,549 matched content/client observations** with March performance features and an April impression outcome.

The inner join means that only content/client combinations observed in both periods are included in this experiment.

In [57]:
df.head()

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,april_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,6787.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255,405.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,8475.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,6091.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.499519,287.0


The resulting rows contain the March decision-period features alongside the April outcome variable. This separation is important: March information represents what could be known before the future outcome, while April information is used only to construct the experimental label or demonstrate leakage.

### Future decline proxy

For this experiment, I define a simple binary future-decline proxy:

- `1` = April impressions were lower than March impressions
- `0` = April impressions were equal to or higher than March impressions

This is a deliberately simple experimental proxy used to demonstrate feature leakage. It is not presented as the official FlyRank target definition.

In [58]:
df["is_declining_label"] = (
    df["april_impressions"] < df["march_impressions"]
).astype(int)


### Proxy label distribution


In [59]:
df["is_declining_label"].value_counts(normalize=True)

,proportion
is_declining_label,
1,0.591483
0,0.408517


The matched dataset contains approximately **59.1% declining observations** and **40.9% non-declining observations** under this experimental definition.

The classes are not perfectly balanced, so accuracy should not be interpreted in isolation. For this small leakage demonstration, the main comparison is between the honest feature set and the deliberately contaminated feature set under the same train/test split.

## Feature Selection

### Legitimate March features

The honest model uses only information available during the March decision period:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

No April-derived information is included in this feature set.

In [60]:
features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

X = df[features].copy()
y = df["is_declining_label"]

## Honest baseline model

In [61]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, pred)

print("Honest accuracy:", honest_accuracy)

Honest accuracy: 0.6341217281614633


I first train a depth-3 decision tree using only the three March GSC performance features. The April outcome is kept completely out of the feature set.

The model achieves **63.41% accuracy** on the held-out test set.

This is the honest reference score for the leakage experiment because the model receives only information that would be available from the March decision period.

## Deliberate leakage test

In [62]:
df["april_impression_change_pct"] = (
    (df["april_impressions"] - df["march_impressions"])
    / df["march_impressions"]
) * 100

In [63]:
leaky_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "april_impression_change_pct"   # DELIBERATE LEAKAGE
]

X_leaky = df[leaky_features].copy()

I now deliberately add `april_impression_change_pct` to the feature set.

This feature is calculated using April impressions, which belong to the future outcome window. It is therefore not knowable at the March decision moment.

More importantly, `april_impression_change_pct` directly contains the relationship used to construct `is_declining_label`: a negative percentage indicates that April impressions are lower than March impressions.

This makes the feature deliberately leaky. The purpose is to demonstrate how a feature derived from future outcome information can artificially inflate a model's evaluation score.

## Leakage result

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, leaky_pred)

print("Leaky accuracy:", leaky_accuracy)

Leaky accuracy: 1.0


I retrain the same depth-3 decision tree using the original March features plus the deliberately leaked `april_impression_change_pct` feature.

The honest model achieved **63.41% held-out accuracy**.

After introducing the future-derived feature, the held-out accuracy increased to **100%**.

This is not evidence that the model became better. The apparent improvement is caused by leakage: the model has access to a feature calculated from the same future information used to define the label.

The experiment demonstrates that a very high evaluation score can be misleading when the feature-generation process violates the decision-time boundary.

For the final feature set, the leaked feature must therefore be removed.

### Leakage removed

In [65]:
# Remove the deliberately leaked feature
X_clean = df[
    [
        "march_impressions",
        "march_clicks",
        "march_avg_position"
    ]
].copy()

print("Leaked feature removed.")
print("Final legitimate features:", X_clean.columns.tolist())

Leaked feature removed.
Final legitimate features: ['march_impressions', 'march_clicks', 'march_avg_position']


The deliberately leaked `april_impression_change_pct` feature is removed.

The final legitimate feature set contains only March GSC performance information:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

The honest accuracy of **63.41%** is retained as the reference result. The 100% score from the contaminated experiment is rejected because it depends on information that would not be available when the decision is made.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# My Response:
The warehouse has uneven data availability across clients and sources. In the March 2026 slice, GSC data was available for 3,611,061 of 9,841,378 rows (36.7%), while GA4 data was available for only 413,966 rows (4.2%). Therefore, GA4-based features cannot be assumed to represent zero engagement when the availability flag is false.

History depth also differs across clients. A single calendar window therefore does not necessarily provide the same amount of historical information for every client, which can affect comparisons across content and clients.

The data also cannot tell me, by itself, whether a page should be refreshed. Performance signals can support prioritization or decision support, but they do not establish the business value of refreshing a page or whether a content change caused a future performance change.

Future-looking labels and features must also be aligned carefully. The leakage experiment demonstrated that introducing April-derived information can produce an artificially high evaluation score. That information must be removed from the legitimate feature set because it would not be available at the March decision moment.

**Named limitation of this slice:**  
This experiment uses March 2026 as the development period and April 2026 as the future outcome window. It therefore does not establish whether the same feature relationships or availability patterns hold across the full warehouse history or across different clients.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.